# Analisis & Evaluasi Hasil Penelitian Temu Kembali Arsip (Bab IV)

**Penelitian:** *Sistem Pencarian Arsip Menggunakan Fuzzy Matching dan Normalisasi Metadata Berbasis Website pada Dinas Perpustakaan dan Kearsipan Kota Palembang*

---

### Catatan Integritas Ilmiah
Notebook ini berfungsi sebagai **penampil analisis dan pelaporan visual** (*consumer viewer*) yang membaca data hasil komputasi di direktori `output/`. Seluruh metrik dihitung secara terisolasi oleh modul mesin evaluasi (`scripts/evaluate.py` dan `scripts/evaluate_extraction_audit.py`) untuk menjaga integritas dan objektivitas data tanpa komputasi manual berulang.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# Jalur berkas hasil komputasi
AUDIT_JSON_PATH = Path("../output/extraction_audit_report.json")
EVAL_JSON_PATH = Path("../output/final_evaluation_results.json")
FIGURES_DIR = Path("../output/figures")

# Fallback jika dieksekusi dari root proyek
if not AUDIT_JSON_PATH.exists() and Path("output/extraction_audit_report.json").exists():
    AUDIT_JSON_PATH = Path("output/extraction_audit_report.json")
    EVAL_JSON_PATH = Path("output/final_evaluation_results.json")
    FIGURES_DIR = Path("output/figures")

print("Status Berkas Output:")
print(f"  - Laporan Audit Ekstraksi : {'Tersedia' if AUDIT_JSON_PATH.exists() else 'Belum Dihasilkan'}")
print(f"  - Laporan Temu Kembali    : {'Tersedia' if EVAL_JSON_PATH.exists() else 'Belum Dihasilkan'}")

## 1. Audit Akurasi Ekstraksi Metadata (100 Sampel Berstrata)

Evaluasi akurasi ekstraksi informasi dari ringkasan teks bebas (`summary_raw`) terhadap 5 entitas metadata: Nomor SPM, Nomor SP2D, Tanggal Dokumen, Nilai Rupiah, dan Nama Pihak (CV/PT). Dilengkapi estimasi interval kepercayaan Wilson score 95%.

In [ ]:
if AUDIT_JSON_PATH.exists():
    with open(AUDIT_JSON_PATH, "r", encoding="utf-8") as f:
        audit_data = json.load(f)

    fields_dict = audit_data.get("fields", {})
    rows = []
    field_labels = {
        "spm": "Nomor SPM",
        "sp2d": "Nomor SP2D",
        "date": "Tanggal Dokumen",
        "amount": "Nilai Rupiah",
        "party": "Nama Pihak (CV/PT)",
    }

    for fld, info in fields_dict.items():
        p_ci = info.get("wilson_ci_ketepatan", [0, 0])
        r_ci = info.get("wilson_ci_keterambilan", [0, 0])
        rows.append({
            "Field Metadata": field_labels.get(fld, fld),
            "Benar": info.get("n_benar"),
            "Salah": info.get("n_salah"),
            "Terlewat": info.get("n_terlewat"),
            "Tidak Ada": info.get("n_tidak_ada_di_teks"),
            "Ketepatan / Precision (%)": round(info.get("ketepatan", 0) * 100, 2),
            "95% CI Ketepatan": f"[{p_ci[0]*100:.1f}%, {p_ci[1]*100:.1f}%]",
            "Keterambilan / Recall (%)": round(info.get("keterambilan", 0) * 100, 2),
            "95% CI Keterambilan": f"[{r_ci[0]*100:.1f}%, {r_ci[1]*100:.1f}%]",
        })

    df_audit = pd.DataFrame(rows)
    display(df_audit)

    # Tampilkan figur visualisasi jika tersedia
    fig1 = FIGURES_DIR / "fig_1_akurasi_ekstraksi_wilson.png"
    if fig1.exists():
        display(Image(filename=str(fig1)))
else:
    print("Berkas extraction_audit_report.json belum ditemukan. Jalankan 'python -m scripts.evaluate_extraction_audit' terlebih dahulu.")

## 2. Hasil Temu Kembali Komparatif (Kelompok Test)

Perbandingan metrik Information Retrieval antara baseline eksak LIKE (e-Arsip SIMANIS) dan empat algoritma Fuzzy Matching berbobot field (Levenshtein, Jaro-Winkler, Token Set, dan N-gram Dice) pada kelompok test set.

In [ ]:
if EVAL_JSON_PATH.exists():
    with open(EVAL_JSON_PATH, "r", encoding="utf-8") as f:
        eval_full = json.load(f)

    manifest = eval_full.get("manifest", {})
    results = eval_full.get("results", {})

    print("Reproducibility Manifest:")
    print(f"  - Commit SHA   : {manifest.get('git_commit_sha')}")
    print(f"  - Testset Hash : {manifest.get('testset_sha256')}")
    print(f"  - Total Kueri  : {manifest.get('n_test_queries')}")
    print(f"  - Baris Ragu   : {manifest.get('total_ragu', 0)}")

    method_rows = []
    for m_key, m_data in results.items():
        ov = m_data.get("overall", {})
        method_rows.append({
            "Metode": m_key,
            "Precision@5": round(ov.get("precision_at_5", 0), 4),
            "Recall@10": round(ov.get("recall_at_10", 0), 4),
            "MRR@10": round(ov.get("mrr_at_10", 0), 4),
            "Hit Rate@5": round(ov.get("hit_rate_at_5", 0), 4),
            "Hit Rate@10": round(ov.get("hit_rate_at_10", 0), 4),
            "Nol Hasil": ov.get("zero_result_queries", 0),
            "Unjudged Top-10": ov.get("unjudged_in_top10", 0),
            "Rata-rata Latensi (ms)": round(ov.get("mean_latency_ms", 0), 2),
        })

    df_eval = pd.DataFrame(method_rows)
    display(df_eval)

    # Tampilkan figur grafik utama
    fig2 = FIGURES_DIR / "fig_2_perbandingan_temu_kembali.png"
    if fig2.exists():
        display(Image(filename=str(fig2)))
else:
    print("Berkas final_evaluation_results.json belum ditemukan. Jalankan 'python -m scripts.evaluate' terlebih dahulu.")

## 3. Analisis Ketahanan Terhadap Variasi Kesalahan Pengetikan (*Noise Robustness*)

Evaluasi kemampuan temu kembali ketika pengguna memasukkan kueri dengan 6 variasi gangguan realistis (*typo*, singkatan, tanda baca/spasi, urutan kata terbalik, kata hilang, ejaan varian).

In [ ]:
if EVAL_JSON_PATH.exists():
    fig3 = FIGURES_DIR / "fig_3_ketahanan_gangguan_noise.png"
    if fig3.exists():
        display(Image(filename=str(fig3)))
    else:
        print("Figur 3 belum diekspor. Jalankan 'python -m scripts.export_report_figures'.")

## 4. Ekspor Ulang Grafik Resolusi Tinggi (300 DPI)

Sel di bawah ini dapat dijalankan kapan saja untuk memperbarui grafik PNG di direktori `output/figures/`.

In [ ]:
from scripts.export_report_figures import export_all_figures

export_all_figures(
    eval_json_path=EVAL_JSON_PATH,
    audit_json_path=AUDIT_JSON_PATH,
    output_dir=FIGURES_DIR,
)
print("[+] Seluruh grafik visual 300 DPI telah diperbarui di:", FIGURES_DIR.resolve())